# Blood Glucose Forecasting: Baselines, Ridge, & XGBoost Benchmark
This notebook implements leakage-safe temporal forecasting across multiple horizons (30, 60, 120 minutes):
1. **Persistence Baseline** ({t+H} = y_t$)
2. **Ridge Regression** (L2-regularized linear model with validation alpha tuning)
3. **XGBoost Regressor** (Gradient boosted decision trees with early stopping)
4. **Systematic Feature Ablation Study**: Testing whether Diurnal Time, Menstrual Cycle, Stress, and Wearables improve forecast performance over pure autoregression.
5. **Preparation for Deep Sequence Models** (LSTM / TFT).

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.forecasting.dataset import ForecastingDataset
from src.forecasting.sequence_builder import SequenceBuilder
from src.forecasting.baseline import PersistenceBaseline
from src.forecasting.ridge_model import RidgeForecaster
from src.forecasting.xgboost_model import XGBoostForecaster
from src.forecasting.evaluation import Evaluator

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Forecasting infrastructure loaded successfully!')

## 1. Load Data & Establish Chronological Partitioning
Partitions are aligned along natural multi-hour sensor disconnect boundaries:
- **Train Partition:** Segments 1–6 (72.1% of data: June 8 – September 27)
- **Validation Partition:** Segment 7 (15.1% of data: September 28 – October 13)
- **Test Partition:** Segment 8 (12.8% of data: October 14 – October 27)
Inter-split gaps: **21 hours 55 minutes** and **38 hours 15 minutes** (zero window leakage).

In [ ]:
# Load features and merge wearable metrics backward
df_wear = pd.read_csv('../data/processed/wearable_metrics.csv')
df = pd.read_csv('../data/processed/glucose_features.csv')

df_wear['dt'] = pd.to_datetime(df_wear['timestamp'])
df['dt'] = pd.to_datetime(df['timestamp'])

hr = df_wear[df_wear['metric_type'] == 'heart_rate'].sort_values('dt')
hrv = df_wear[df_wear['metric_type'] == 'hrv_sdnn'].sort_values('dt')

df = pd.merge_asof(df, hr[['dt', 'value']].rename(columns={'value': 'heart_rate_bpm'}), on='dt', direction='backward', tolerance=pd.Timedelta('30min'))
df = pd.merge_asof(df, hrv[['dt', 'value']].rename(columns={'value': 'hrv_sdnn_ms'}), on='dt', direction='backward', tolerance=pd.Timedelta('60min'))
df.drop(columns=['dt'], inplace=True)

ds = ForecastingDataset()
ds.df = df

split = ds.create_segment_split(
    train_segments=[1, 2, 3, 4, 5, 6],
    val_segments=[7],
    test_segments=[8]
)

b = split.boundaries
pd.DataFrame({
    'Partition': ['Train', 'Validation', 'Test'],
    'Segment IDs': ['1, 2, 3, 4, 5, 6', '7', '8'],
    'Record Count': [b['train_rows'], b['val_rows'], b['test_rows']],
    'Percentage': [f"{b['train_fraction']*100:.1f}%", f"{b['val_fraction']*100:.1f}%", f"{b['test_fraction']*100:.1f}%"],
    'Start Timestamp': [b['train_start'], b['val_start'], b['test_start']],
    'End Timestamp': [b['train_end'], b['val_end'], b['test_end']]
})

## 2. Benchmark Experiments Summary Table
Comparing **Persistence Baseline**, **Ridge Regression**, and **XGBoost** across all 3 horizons (30m, 60m, 120m).

In [ ]:
df_exp = pd.read_csv('../results/feature_ablation_study.csv')
cols_show = ['model_name', 'feature_set', 'horizon_minutes', 'mae', 'rmse', 'r2', 'clinical_accuracy_pct']
df_exp[cols_show]

## 3. Visualization: Model Performance vs. Forecast Horizon
Tracks how prediction error scales with lookahead horizon for each model family.

In [ ]:
plt.figure(figsize=(12, 5))

# Filter best performing configuration per model
p_df = df_exp[df_exp['model_name'] == 'Persistence']
r_df = df_exp[(df_exp['model_name'] == 'Ridge') & (df_exp['feature_set'] == 'FS3_Glucose_Time_Cycle')]
x_df = df_exp[(df_exp['model_name'] == 'XGBoost') & (df_exp['feature_set'] == 'FS3_Glucose_Time_Cycle')]

horizons = [30, 60, 120]
plt.subplot(1, 2, 1)
plt.plot(p_df['horizon_minutes'], p_df['mae'], 'o--', color='gray', linewidth=2, label='Persistence Baseline')
plt.plot(r_df['horizon_minutes'], r_df['mae'], 's-', color='#2b83ba', linewidth=2, label='Ridge (Time + Cycle)')
plt.plot(x_df['horizon_minutes'], x_df['mae'], '^-', color='#d7191c', linewidth=2.5, label='XGBoost (Time + Cycle)')
plt.title('Mean Absolute Error (MAE) by Horizon', fontsize=12, fontweight='bold')
plt.xlabel('Forecast Horizon (Minutes)')
plt.ylabel('MAE (mg/dL)')
plt.xticks(horizons, ['30 min', '60 min', '120 min'])
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(p_df['horizon_minutes'], p_df['r2'], 'o--', color='gray', linewidth=2, label='Persistence')
plt.plot(r_df['horizon_minutes'], r_df['r2'], 's-', color='#2b83ba', linewidth=2, label='Ridge')
plt.plot(x_df['horizon_minutes'], x_df['r2'], '^-', color='#d7191c', linewidth=2.5, label='XGBoost')
plt.axhline(0, color='black', linestyle=':', alpha=0.5)
plt.title('Coefficient of Determination (R²) by Horizon', fontsize=12, fontweight='bold')
plt.xlabel('Forecast Horizon (Minutes)')
plt.ylabel('R² Score')
plt.xticks(horizons, ['30 min', '60 min', '120 min'])
plt.legend()

plt.tight_layout()
plt.show()

## 4. Feature Ablation: Do Cycle, Stress, & Wearables Help?
Testing whether adding Diurnal, Menstrual Cycle, Stress, and Wearable features improves test MAE over pure autoregression.

In [ ]:
pivot_ablation = df_exp[df_exp['model_name'] == 'XGBoost'].pivot(
    index='feature_set',
    columns='horizon_minutes',
    values='mae'
)
pivot_ablation.columns = ['30 min MAE (mg/dL)', '60 min MAE (mg/dL)', '120 min MAE (mg/dL)']
pivot_ablation

## 5. Next Step: Sequence Deep Learning (LSTM & TFT)
The baselines and feature importance profiles are established.
We are now ready to implement:
- **Univariate & Multi-variate LSTM** (capturing multi-step non-linear temporal dependencies)
- **Temporal Fusion Transformer (TFT)** (interpreting static cycle metadata vs dynamic glucose/wearable temporal inputs).